# Explainable AI for Transformer-Based Satellite Change Detection (LEVIR-CD)
### Course Project: Information Visualization (InfoVis)

This Jupyter Notebook provides the training analysis, model comparison, qualitative predictions, and Explainable AI (XAI) diagnostics for the Satellite Image Change Detection project on the LEVIR-CD dataset.

## Project Goals
1. Build a **Siamese Swin Transformer** change detection network and compare it with a **Siamese CNN Baseline**.
2. Visualize bi-temporal change predictions using **aligned visual comparison grids** and **color-coded error difference maps**.
3. Apply and evaluate **Swin Attention Rollout** and **Grad-CAM** visual explanations.
4. Format metric logs into a flat structure (`dashboard_data.csv`) for direct import into **Tableau** and **Power BI**.

### 1. Load Dependencies & Reproducibility Settings

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Set display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Paths
project_dir = r"d:/vit/SEMESTER-5/IV/DA"
outputs_dir = os.path.join(project_dir, 'outputs')

print("Directories verified. Ready to load metrics and visualizations!")

### 2. Environment & Experiment Configuration

In [ ]:
config_path = os.path.join(outputs_dir, 'experiment_config.json')
with open(config_path, 'r') as f:
    exp_config = json.load(f)

print("--- Experiment Metadata & Parameters ---")
for k, v in exp_config.items():
    print(f"{k:30}: {v}")

### 3. Quantitative Test Metrics (Model Comparison)
Let's load the saved test evaluation report comparing the **Siamese Swin Transformer** vs the **Siamese CNN Baseline**.

In [ ]:
metrics_csv = os.path.join(outputs_dir, 'metrics', 'test_metrics.csv')
df_metrics = pd.read_csv(metrics_csv, index_col=0)
df_metrics

### 4. Training & Validation Curves
Here, we plot the saved loss and metric curves to analyze convergence and overfitting for both models.

In [ ]:
curves_img_path = os.path.join(outputs_dir, 'visualizations', 'training_curves.png')
bar_img_path = os.path.join(outputs_dir, 'visualizations', 'precision_recall_f1_comparison.png')

# Display loss and metric curves
plt.figure(figsize=(15, 7))
plt.imshow(Image.open(curves_img_path))
plt.axis('off')
plt.title('Training & Validation Curves', fontsize=14, fontweight='bold')
plt.show()

# Display Precision/Recall/F1 bar chart comparison
plt.figure(figsize=(12, 9))
plt.imshow(Image.open(bar_img_path))
plt.axis('off')
plt.title('Precision, Recall, and F1 Score Comparison', fontsize=14, fontweight='bold')
plt.show()

### 5. Change Prediction Grids (T1 | T2 | GT Mask | Pred Mask | Difference Map)
Let's visualize the actual predictions. The **Difference Map** uses standard segmentation error colors:
- **Green**: True Positives (TP - correct changes)
- **Red**: False Positives (FP - background falsely predicted as changed)
- **Blue**: False Negatives (FN - actual changes missed by the model)
- **Grey**: True Negatives (TN - background correctly identified)

In [ ]:
pred_dir = os.path.join(outputs_dir, 'predictions')
pred_images = [f for f in os.listdir(pred_dir) if f.endswith('.png')]

for img_name in sorted(pred_images)[:2]: # Show first 2 comparison samples
    img_path = os.path.join(pred_dir, img_name)
    plt.figure(figsize=(18, 9))
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.title(f"Aligned Prediction Grid: {img_name}", fontsize=12, fontweight='bold')
    plt.show()

### 6. Explainable AI Overlays (Attention Rollout vs Grad-CAM)
This section displays the visual explanations: 
- **Swin Attention Rollout** (Stage 4 window attention merged recursively)
- **Grad-CAM** (decoder features mapped by classification gradients)

The heatmaps highlight which areas of the satellite images drove the model's change predictions.

In [ ]:
xai_dir = os.path.join(outputs_dir, 'xai')
xai_images = [f for f in os.listdir(xai_dir) if f.endswith('.png')]

for img_name in sorted(xai_images)[:2]:
    img_path = os.path.join(xai_dir, img_name)
    plt.figure(figsize=(18, 6))
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.title(f"XAI Diagnostics Overlay Grid: {img_name}", fontsize=12, fontweight='bold')
    plt.show()

### 7. Explanation Quality & Stability Analysis
We evaluate explanation quality programmatically:
- **Explanation IoU (XAI Overlap)**: Measure of overlap between explanation regions (heatmap > 0.5) and the Ground Truth.
- **Stability (XAI Stability)**: Cosine similarity between original heatmaps and heatmaps from inputs with small random noise ($\sigma = 0.05$).

In [ ]:
dashboard_csv = os.path.join(outputs_dir, 'metrics', 'dashboard_data.csv')
df_dashboard = pd.read_csv(dashboard_csv)

# Display sample-level dataframe
print("--- Sample-level Performance & Explanations dataset ---")
display(df_dashboard.head(10))

# Summary statistics for Swin vs Baseline explanations
print("\n--- XAI Explanation Quality Comparison ---")
xai_summary = df_dashboard.groupby('model_type')[['xai_overlap', 'xai_stability']].mean()
display(xai_summary)

### 8. Statistical Distributions
Displaying the statistical overview plots created for your InfoVis presentation.

In [ ]:
dist_img_path = os.path.join(outputs_dir, 'visualizations', 'statistical_distributions.png')

plt.figure(figsize=(18, 7))
plt.imshow(Image.open(dist_img_path))
plt.axis('off')
plt.title('InfoVis Dataset Distributions Overview', fontsize=14, fontweight='bold')
plt.show()

### 9. Visualizing in Tableau / Power BI
The data to load into your dashboard is stored at:
**`d:/vit/SEMESTER-5/IV/DA/outputs/metrics/dashboard_data.csv`**

#### Recommended Tableau Dashboard Layout:
1. **Performance Panel**: Use a side-by-side bar chart showing **F1-score**, **IoU**, and **Accuracy** by `model_type`.
2. **Prediction Scatter**: Plot `change_percentage_ground_truth` vs `change_percentage_prediction`, colored by `model_type` with a diagonal trendline to show deviation.
3. **XAI Performance Bubble**: Bubble plot mapping `xai_overlap` (X), `f1` (Y), and sized by `xai_stability` to demonstrate that models with high explanation stability and overlap correspond to stronger overall segmentation scores.